# 다이캐스팅 공정 데이터 기반 품질 예측 분석

## 프로젝트 개요
다이캐스팅은 융융 금속을 금형에 고압으로 주입하여 정밀한 제품을 생산하는 공정입니다.
생산 효율을 높이고 불량률을 낮추기 위해 공정변수(주조 온도, 압력, 속도 등) 및 센서 데이터를 분석하여 불량 유형을 자동 예측하는 머신러닝 모델을 개발합니다.

## 비즈니스 문제 정의
### 현재 상황
- 불량(미성형, 박리, 기공, 평탄, 개재물 등)이 발생해도 **육안 검사에 의존**하여 판정 기준의 주관성과 검사 속도의 한계로 인해 생산성이 저하됩니다.
- 불량 발생 원인을 추적하기 어려워 **공정 개선 및 문제 해결이 어렵습니다.**
- 공정 데이터와 품질 검사를 효과적으로 매핑하지 못해 **실시간 품질 관리 및 재발 방지 대책이 부족**합니다.

### 해결 필요성
- 공정변수 및 센서 데이터와 불량 유형 발생 간의 관계 파악 필요
- 불량 유형을 자동 분류하는 모델 개발로 품질 예측이 가능한 시스템 구축 필요
- 불량 발생 주요 원인을 분석하여 공정 최적화를 위한 인사이트 도출

## 분석 목표 및 KPI 설정
### 분석 목표
1. 다이캐스팅 공정에서 발생하는 다양한 불량 유형(미성형, 박리, 기공, 평탄, 개재물 등)을 자동 예측하는 머신러닝 모델을 개발
2. 공정 데이터(주조 압력, 금형 온도, 주입 속도 등)와 센서 데이터(온도, 압력, 유량, 진동 등)를 활용하여 불량 여부를 판별
4. 실시간 품질 예측 체계를 구축하여 조기 경보 시스템 도입 및 불량률 감소

### 비즈니스 KPI
- 현재 불량률 대비 불량률 10% 감소
- 불량 예측 모델의 정확도 90% 이상






In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import skew, kurtosis
import warnings

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

In [ ]:
df = pd.read_csv('../../data/DieCasting_Quality_Raw_Data.csv', header=[0, 1])

print("="*60)
print("데이터셋 로드 완료!")
print("="*60)

print(df.shape)
df.info()

In [ ]:
# 데이터셋 샘플 확인
print("\n" + "="*60)
print("DieCasting 샘플 데이터")
print("="*60)
display(df.head())

In [ ]:
# 원본 데이터 기초 통계 확인
print("\n" + "="*60)
print("다이캐스팅 데이터셋 기초 통계")
print("="*60)
display(df.describe(include='all').T)

## 2. 데이터 전처리

### 2-1. 컬럼명 공백 제거

In [ ]:
df.rename(columns={'Biscuit_Thickness ': 'Biscuit_Thickness', 'Clamping_Force ': 'Clamping_Force', ' Pressure_Rise_Time': 'Pressure_Rise_Time'}, level=1, inplace=True)

### 2-2. 컬럼명 소문자 전환

In [ ]:
df.columns = df.columns.map(lambda x: tuple(str(level).lower() for level in x))

### 2-3. 불필요한 컬럼 제거
id 값은 데이터와 무관하다고 판단하기 때문에 제거

In [ ]:
df.drop(columns=[('process', 'id'),('process', 'shot')], inplace=True)
print("="*60)
print("id 컬럼 제거 완료!")
print("="*60)

In [ ]:
print(df.shape)
print(df.info())

### 2-3. 중복 데이터 확인

id 값을 제외하고 모든 컬럼의 값이 동일한 중복 행 제거

In [ ]:
### 2. 중복 제거
# 모든 남아있는 컬럼을 기준으로 중복 제거
initial_row_count = len(df)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
final_row_count = len(df)

# 결과 확인
print(f"제거 전 행 개수: {initial_row_count}")
print(f"제거 후 행 개수: {final_row_count}")
print(f"삭제된 중복 행 개수: {initial_row_count - final_row_count}")

### 2-4. 결측치 확인
몇 개의 컬럼에서 결측치가 확인되지만 학습/정답 데이터 분리를 위해 우선은 그대로 유지

In [ ]:
missing_process_sensor = df.loc[:, (['process', 'sensor'], slice(None))].isna().sum()
missing_process_sensor = missing_process_sensor[missing_process_sensor > 0].reset_index()
missing_process_sensor.columns = ['', 'Columns', 'Missing Count']
display(missing_process_sensor)

### 2-5 파생컬럼 생성

In [ ]:
defect_cols = [col for col in df.columns if col[0]=='defects']

df[('defect_flag','is_defect')] = (df[defect_cols] == 1).any(axis=1).astype(int)

df[('defect_flag','is_defect')].value_counts() # --> 0: 정상, 1: 불량

## 3. 데이터 시각화

### 3-1. 데이터의 히스토그램 확인
- 히스토그램에서 몇몇 데이터의 이봉분포 확인됨
- 상품 유형에 따른 데이터 차이가 발생하기 때문으로 확인
- 상품 유형 별로 데이터를 분리한 분석 진행이 필요하다고 판단

In [ ]:
plt.figure(figsize=(30,30))
plt.subplots_adjust(hspace=0.38)
# 각 변수의 막대그래프 개수
for index,value in enumerate(df):
 sub=plt.subplot(12,5,index+1)
 sub.hist(df[value],facecolor=(144/255,171/255,221/255),
	 	 	 linewidth=.3,edgecolor='black')
 plt.title(value)

In [ ]:
# product_type 별 히스토그램 시각화
product_type_1 = df[df[('process', 'product_type')] == 1]
product_type_2 = df[df[('process', 'product_type')] == 2]

num_cols = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
num_total = len(num_cols)

n_cols_per_row = 3
n_rows = (num_total + n_cols_per_row - 1) // n_cols_per_row

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols_per_row, figsize=(n_cols_per_row * 6, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(product_type_1[col].dropna(), bins=50, alpha=0.6, edgecolor='black', linewidth=0.3, label='type 1')
    axes[i].hist(product_type_2[col].dropna(), bins=50, alpha=0.6, edgecolor='black', linewidth=0.3, label='type 2')
    axes[i].set_title(f'{col[0]}_{col[1]}')
    axes[i].legend()

for j in range(num_total, len(axes)):
    fig.delaxes(axes[j])

plt.subplots_adjust(hspace=0.4)
plt.show()

In [ ]:
# process, sensor 컬럼 박스플롯 (product_type 1, 2 비교)
product_type_1 = df[df[('process', 'product_type')] == 1]
product_type_2 = df[df[('process', 'product_type')] == 2]

ps_cols = [(lvl0, lvl1) for lvl0, lvl1 in df.columns if lvl0 in ['process', 'sensor']]
num_total = len(ps_cols)

n_cols_per_row = 3
n_rows = (num_total + n_cols_per_row - 1) // n_cols_per_row

fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols_per_row, figsize=(n_cols_per_row * 6, n_rows * 4))
axes = axes.flatten()

for i, (lvl0, lvl1) in enumerate(ps_cols):
    data = [product_type_1[(lvl0, lvl1)].dropna(), product_type_2[(lvl0, lvl1)].dropna()]
    bp = axes[i].boxplot(data, labels=['type 1', 'type 2'], patch_artist=True)
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][1].set_facecolor('coral')
    axes[i].set_title(f'{lvl0}_{lvl1}')
    axes[i].set_ylabel('값')
    axes[i].grid(True, alpha=0.3)

for j in range(num_total, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

# Min/ Max/product type 컬럼 제거

In [ ]:
product_type_1 = product_type_1.drop(columns=[('process','product_type'), ('sensor', 'air_pressure_min'), ('sensor', 'air_pressure_max'), 
                                              ('sensor', 'coolant_temp_min'), ('sensor', 'coolant_temp_max'), 
                                              ('sensor', 'factory_temp_min'), ('sensor', 'factory_temp_max'), 
                                              ('sensor', 'factory_humidity_min'), ('sensor', 'factory_humidity_max')])
product_type_2 = product_type_2.drop(columns=[('process','product_type'), ('sensor', 'air_pressure_min'), ('sensor', 'air_pressure_max'),
                                              ('sensor', 'coolant_temp_min'), ('sensor', 'coolant_temp_max'),
                                              ('sensor', 'factory_temp_min'), ('sensor', 'factory_temp_max'), 
                                              ('sensor', 'factory_humidity_min'), ('sensor', 'factory_humidity_max')])

In [ ]:
product_type_1.info()

## 3. 이상치 확인 및 처리


### 3-1. 불량 유형 여부 이상치
불량 유형에 해당되는 1(불량), 0(정상) 값을 벗어나는 2 또는 3과 같은 값을 1(불량) 값으로 대체




In [ ]:
### 3. 2,3을 불량 1로 대체
product_dfs = [product_type_1, product_type_2]
product_names = ['Type 1', 'Type 2']

for product_df, product_name in zip(product_dfs, product_names):
    defect_cols = product_df['defects'].columns
    
    for col in defect_cols:
        product_df.loc[
            product_df[('defects', col)] >= 2,
            ('defects', col)
        ] = 1
    
    print(f"{product_name} 이상치 값 대체 완료!")

In [ ]:

print("="*60)
print("각 타입별로 0밖에 없는 defect 컬럼 제거 전:")
print("="*60)
print(f"type1: {product_type_1.shape}")
print(f"type2: {product_type_2.shape}")

# 각 타입별로 0밖에 없는 defect 컬럼 출력
for name, df_type in [('type1', product_type_1), ('type2', product_type_2)]:
    defect_cols = df_type['defects'].columns
    zero_cols = [col for col in defect_cols if df_type[('defects', col)].sum() == 0]
    print(f"=== {name} - 0밖에 없는 불량 컬럼 ===")
    if zero_cols:
        for col in zero_cols:
            print(f"  - {col}")
    else:
        print("  없음")
    print()

# 각 타입별로 0밖에 없는 defect 컬럼 제거
def drop_zero_defect_cols(df_type):
    defect_cols = df_type['defects'].columns
    zero_cols = [col for col in defect_cols if df_type[('defects', col)].sum() == 0]
    print(f"제거 컬럼: {zero_cols}")
    return df_type.drop(columns=[('defects', c) for c in zero_cols])

product_type_1 = drop_zero_defect_cols(product_type_1)
product_type_2 = drop_zero_defect_cols(product_type_2)

print("="*60)
print("각 타입별로 0밖에 없는 defect 컬럼 제거 후:")
print("="*60)
print(f"type1: {product_type_1.shape}")
print(f"type2: {product_type_2.shape}")

In [ ]:
### 3-1. 불량 유형별 불량률 확인
# Product_Type별 불량 컬럼 불량률 확인
type_dfs   = [product_type_1, product_type_2]
type_names = ['Type 1', 'Type 2']

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

for ax, type_df, type_name in zip(axes, type_dfs, type_names):
    defect_cols = type_df['defects'].columns
    defect_rates = {col: type_df[('defects', col)].mean() * 100 for col in defect_cols if type_df[('defects', col)].any()}
    defect_rates = pd.Series(defect_rates).sort_values(ascending=False)

    sns.barplot(x=defect_rates.index, y=defect_rates.values, palette='viridis', ax=ax)
    ax.set_title(f'[{type_name}] 불량 컬럼별 불량률 (%)', fontsize=14, fontweight='bold')
    ax.set_xlabel('불량 컬럼', fontsize=11)
    ax.set_ylabel('불량률 (%)', fontsize=11)
    ax.set_xticks(range(len(defect_rates)))
    ax.set_xticklabels(defect_rates.index, rotation=45, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f'{height:.2f} %',
            (p.get_x() + p.get_width() / 2, height),
            ha='center',
            va='bottom',
            fontsize=9
        )

plt.tight_layout()
plt.show()
# 전체 정상/불량률 확인
# 레이아웃: 행=Type, 열=[파이차트 | 바차트]
type_dfs   = [product_type_1, product_type_2]
type_names = ['Type 1', 'Type 2']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, (type_df, type_name) in enumerate(zip(type_dfs, type_names)):
    is_defect = type_df['defects'].any(axis=1)
    defect_count = is_defect.sum()
    normal_count = (~is_defect).sum()
    total = len(type_df)

    # 파이차트 (col 0)
    axes[i, 0].pie(
        [normal_count, defect_count],
        labels=['정상', '불량'],
        autopct='%1.1f%%',
        startangle=90,
        counterclock=False,
        colors=['steelblue', 'tomato'],
    )
    axes[i, 0].set_title(f'[{type_name}] 정상/불량 비율\n(전체 {total}행)', fontsize=12, fontweight='bold')

    # 바차트 (col 1)
    axes[i, 1].bar(['정상', '불량'], [normal_count, defect_count], color=['steelblue', 'tomato'], alpha=0.8)
    axes[i, 1].set_title(f'[{type_name}] 정상/불량 건수', fontsize=12, fontweight='bold')
    axes[i, 1].set_ylabel('건수')
    for j, v in enumerate([normal_count, defect_count]):
        axes[i, 1].text(j, v + total * 0.005, f'{v}\n({v/total*100:.1f}%)', ha='center', fontsize=10)
    axes[i, 1].grid(axis='y', alpha=0.3)

plt.suptitle('Product_Type별 정상/불량률', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 불량 컬럼 범주화 과정

In [ ]:
# -----------------------------
# 그룹 정의
# -----------------------------
defect_groups_1 = {
    "표면": ["dent_1", "stain_1", "exfoliation_1", "exfoliation_2"],
    "구조": ["short_shot_1", "short_shot_2", "bubble_1", "bubble_2", "deformation_1", "deformation_2"],
}

defect_groups_2 = {
    "표면": [
        "stain_1",
        "dent_1",
        "dent_2",
        "scratch_1",
        "buring_mark_1"
    ],

    "구조": [
        "short_shot_1",
        "short_shot_2",
        "blow_hole_1",
        "blow_hole_2",
        "bubble_1",
        "crack_1",
        "crack_2"
    ],

    "이물질": [
        "contamination_1",
        "contamination_2",
        "impurity_1",
        "impurity_2",
        "inclusions_2"
    ]
}


# -----------------------------
# 함수: 그룹별 개수 계산
# -----------------------------
def get_group_counts(product_df, defect_groups):

    defect_df = product_df['defects']
    group_counts = {}

    for group, cols in defect_groups.items():
        usable = [c for c in cols if c in defect_df.columns]

        group_counts[group] = (
            defect_df[usable].any(axis=1).sum()
            if usable else 0
        )

    group_counts = {k: v for k, v in group_counts.items() if v > 0}

    return group_counts


counts_1 = get_group_counts(product_type_1, defect_groups_1)
counts_2 = get_group_counts(product_type_2, defect_groups_2)


# -----------------------------
# pastel 색상
# -----------------------------
colors = ["#FFB3BA", "#BAE1FF", "#BAFFC9", "#FFFFBA"]


# -----------------------------
# subplot 생성
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].pie(
    counts_1.values(),
    labels=counts_1.keys(),
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False,
    colors=colors,
    textprops={"fontsize": 13, "fontweight": "bold"}
)

axes[0].set_title(
    'Product Type 1 - 불량 그룹 분포',
    fontsize=13,
    fontweight='bold'
)


axes[1].pie(
    counts_2.values(),
    labels=counts_2.keys(),
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False,
    colors=colors,
    textprops={"fontsize": 13, "fontweight": "bold"}
)

axes[1].set_title(
    'Product Type 2 - 불량 그룹 분포',
    fontsize=13,
    fontweight='bold'
)


plt.tight_layout()
plt.show()

# Train / Test 분리 후 eda 시작

In [ ]:
# 라이브러리 Import
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("="*60)

# **타입별 분석**

## **Type 1**

In [ ]:
# 결측치 확인
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': product_type_1.isnull().sum(),
    '결측비율(%)': (product_type_1.isnull().sum() / len(product_type_1) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

### Train/Test 분리

In [ ]:
def get_train_test_data(product_type, defect_groups, test_size=0.2, random_state=42):

    # X 준비
    X = pd.concat([product_type['process'], product_type['sensor']], axis=1)
    X.columns = (
        [f"process_{col}" for col in product_type['process'].columns] +
        [f"sensor_{col}"  for col in product_type['sensor'].columns]
    )

    # y 준비
    defects_data = product_type['defects']
    y = pd.DataFrame(index=defects_data.index)
    for group, cols in defect_groups.items():
        usable = [c for c in cols if c in defects_data.columns]
        y[group] = defects_data[usable].max(axis=1) if usable else 0

    # stratify 기준 생성
    strata = y.astype(str).agg("".join, axis=1)

    # train/test 분리
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=strata
    )

    # 결과 출력
    print(f"Train set: {X_train.shape} / Test set: {X_test.shape}")
    print("\n[불량 유형 비율(%)]")
    display((y.mean() * 100).round(2).to_frame("rate_%"))
    print("\n[Train set 타겟 분포]")
    print(y_train.value_counts())
    print("\n[Test set 타겟 분포]")
    print(y_test.value_counts())

    return X_train, X_test, y_train, y_test


defect_groups_1 = {
    "표면": ["dent_1", "stain_1", "exfoliation_1", "exfoliation_2"],
    "구조": ["short_shot_1", "short_shot_2", "bubble_1", "bubble_2", "deformation_1", "deformation_2"],
}

X_train, X_test, y_train, y_test = get_train_test_data(product_type_1, defect_groups_1)

In [ ]:
print(X_train.info())
print("=" * 60)
print(y_train.info())

In [ ]:
plt.subplots(figsize=(30,30))
sns.heatmap(data=X_train.corr(),linewidth=0.1,annot=True,fmt='.2f',cmap='coolwarm')

## 상관계수가 1인 두 변수 중 하나를 제거
- cylinder_pressure 제거하기

In [ ]:
product_type_1 = product_type_1.drop(columns=[('process','cylinder_pressure')])
print(product_type_1.shape)
print(product_type_1.columns)

In [ ]:
# Product_Type이 2인 행만 필터링하여 저장
product_type_1.to_csv('../data_processed/product_type_1.csv', index=False, encoding='utf-8-sig')

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_data(product_type_1, defect_groups_1)

# 단변량 분석
- 변수 하나의 분포/특성 파악


In [ ]:
print("=" * 60)
print("독립변수 분포 분석")
print("=" * 60)

import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

# 수치형 변수만 선택
num_cols = X_train.select_dtypes(include='number').columns

print(f"\n[수치형 변수 개수] {len(num_cols)}개")
print(num_cols.tolist())

# 각 변수의 기술통계 + 왜도 + 첨도 출력
summary_list = []

for col in num_cols:
    data = X_train[col].dropna()
    
    summary_list.append({
        '변수명': col,
        'count': data.count(),
        'mean': data.mean(),
        'std': data.std(),
        'min': data.min(),
        '25%': data.quantile(0.25),
        '50%': data.median(),
        '75%': data.quantile(0.75),
        'max': data.max(),
        '왜도': skew(data),
        '첨도': kurtosis(data)
    })

summary_df = pd.DataFrame(summary_list)
#print("\n[독립변수 통계 요약]")
#display(summary_df.round(3))

# 시각화
n_cols = 4   # 한 행에 변수 2개 (각 변수당 hist + box)
n_rows = math.ceil(len(num_cols) / (n_cols // 2))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

idx = 0

for col in num_cols:
    data = X_train[col].dropna()
    
    # 히스토그램
    sns.histplot(data, bins=30, ax=axes[idx], color='steelblue', kde=True)
    axes[idx].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
                      label=f"mean: {data.mean():.2f}")
    axes[idx].axvline(data.median(), color='green', linestyle='--', linewidth=1.5,
                      label=f"median: {data.median():.2f}")
    axes[idx].set_title(f"{col} hist\n왜도={skew(data):.2f}, 첨도={kurtosis(data):.2f}", fontsize=12)
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

    # 박스플롯
    sns.boxplot(x=data, ax=axes[idx + 1], color='orange')
    axes[idx + 1].set_title(f"{col} box", fontsize=12)
    axes[idx + 1].grid(True, alpha=0.3)

    idx += 2

# 남는 subplot 제거
for j in range(idx, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


---

## 왜도가 큰 변수

다음과 같은 특징이 관찰됨

- 한쪽으로 치우친 분포
- 긴 꼬리(long tail)
- 박스플롯에서 이상치 다수 존재

예시 변수

- velocity_1
- velocity_2
- high_velocity
- rapid_rise_time


In [ ]:
skew_vals = X_train.skew().abs()

log_cols = skew_vals[skew_vals > 3].index

for c in log_cols:
    X_train[c] = np.log1p(X_train[c])
    X_test[c]  = np.log1p(X_test[c])

# 다변량 분석
- 변수 여러 개 간의 관계 분석


In [ ]:
print("\n" + "="*60)
print("EDA용 데이터 생성")
print("="*60)

train_eda = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
)

print("train_eda shape:", train_eda.shape)
display(train_eda.head())
print("\n컬럼 목록:")
print(train_eda.columns.tolist())

## 통계적 가설 검정 (t-test / ANOVA)

EDA 과정에서 변수와 불량 발생 사이의 관계를 확인하기 위해
boxplot, 중앙값 비교, 구간별 집계 분석 등을 수행하였다.

하지만 이러한 시각적 비교만으로는 차이가 실제로 의미 있는지,
즉 통계적으로 유의한 차이인지 판단하기 어렵다.

따라서 각 공정 변수에 대해
불량 발생 여부에 따라 값의 평균이 실제로 다른지
통계적 검정을 수행한다.

이번 분석에서는 다음을 수행한다.

1. 각 공정 변수 vs 표면 불량 → t-test
2. 각 공정 변수 vs 구조 불량 → t-test
3. 변수 구간 vs 불량률 → ANOVA
4. 유의한 변수 찾기 (p-value 기준)

기준:

p-value < 0.05 → 유의함
p-value < 0.01 → 매우 유의함
p-value < 0.001 → 강한 관계

In [ ]:
from scipy.stats import ttest_ind

print("\n" + "="*60)
print("전체 변수 자동 t-test")
print("="*60)

# --------------------------------------------------
# 1. EDA용 데이터 준비
# --------------------------------------------------
train_eda = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
).copy()

# 수치형 독립변수 추출
all_num_cols = train_eda.select_dtypes(include="number").columns.tolist()
target_cols = ["표면", "구조"]
feature_cols = [c for c in all_num_cols if c not in target_cols]

print(f"분석 대상 변수 개수: {len(feature_cols)}")
print(feature_cols)


# --------------------------------------------------
# 2. 단일 타겟에 대한 t-test 함수
# --------------------------------------------------
def run_ttest_all_features(df, feature_cols, target_col):
    results = []

    for col in feature_cols:
        temp = df[[col, target_col]].dropna().copy()

        # 그룹 분리
        g0 = temp[temp[target_col] == 0][col]
        g1 = temp[temp[target_col] == 1][col]

        # 그룹 크기 체크
        if len(g0) < 2 or len(g1) < 2:
            continue

        # Welch's t-test (등분산 가정 안 함)
        t_stat, p_val = ttest_ind(g0, g1, equal_var=False)

        mean_0 = g0.mean()
        mean_1 = g1.mean()
        median_0 = g0.median()
        median_1 = g1.median()

        results.append({
            "feature": col,
            "target": target_col,
            "n_0": len(g0),
            "n_1": len(g1),
            "mean_0": mean_0,
            "mean_1": mean_1,
            "mean_diff": mean_1 - mean_0,
            "median_0": median_0,
            "median_1": median_1,
            "median_diff": median_1 - median_0,
            "t_stat": t_stat,
            "p_value": p_val
        })

    result_df = pd.DataFrame(results)

    # 유의성 라벨
    def sig_label(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    result_df["significance"] = result_df["p_value"].apply(sig_label)
    result_df["is_significant_0.05"] = result_df["p_value"] < 0.05

    return result_df.sort_values("p_value", ascending=True).reset_index(drop=True)


# --------------------------------------------------
# 3. 표면 / 구조 각각 수행
# --------------------------------------------------
surface_ttest = run_ttest_all_features(train_eda, feature_cols, "표면")
structure_ttest = run_ttest_all_features(train_eda, feature_cols, "구조")

print("\n[표면 불량 기준 t-test 결과]")
display(surface_ttest)

print("\n[구조 불량 기준 t-test 결과]")
display(structure_ttest)


# --------------------------------------------------
# 4. 유의한 변수만 보기
# --------------------------------------------------
surface_sig = surface_ttest[surface_ttest["p_value"] < 0.05].copy()
structure_sig = structure_ttest[structure_ttest["p_value"] < 0.05].copy()

print("\n[표면 불량 기준 유의한 변수 (p < 0.05)]")
display(surface_sig)

print("\n[구조 불량 기준 유의한 변수 (p < 0.05)]")
display(structure_sig)


# --------------------------------------------------
# 차이 큰 순으로 정렬
# mean_diff, median_diff 둘 다 고려
# --------------------------------------------------
surface_t_sig_sorted = surface_sig.sort_values(
    ["median_diff", "mean_diff", "p_value"],
    ascending=[False, False, True]
).reset_index(drop=True)

structure_t_sig_sorted = structure_sig.sort_values(
    ["median_diff", "mean_diff", "p_value"],
    ascending=[False, False, True]
).reset_index(drop=True)

# --------------------------------------------------
# 표면/구조에서 공통적으로 유의한 변수
# --------------------------------------------------
surface_sig_features = set(surface_t_sig_sorted["feature"])
structure_sig_features = set(structure_t_sig_sorted["feature"])

common_features = sorted(
    surface_sig_features & structure_sig_features
)

print("\n[표면/구조 공통적으로 유의한 변수]")
display(common_features)

In [ ]:
from scipy.stats import mannwhitneyu

print("\n" + "="*60)
print("전체 변수 자동 Mann–Whitney U test")
print("="*60)

# --------------------------------------------------
# 1. EDA용 데이터
# --------------------------------------------------
train_eda = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
).copy()

all_num_cols = train_eda.select_dtypes(include="number").columns.tolist()
target_cols = ["표면", "구조"]
feature_cols = [c for c in all_num_cols if c not in target_cols]

print(f"분석 대상 변수 개수: {len(feature_cols)}")
print(feature_cols)


# --------------------------------------------------
# 2. Mann–Whitney U test 함수
# --------------------------------------------------
def run_mwu_all_features(df, feature_cols, target_col):
    results = []

    for col in feature_cols:
        temp = df[[col, target_col]].dropna().copy()

        g0 = temp[temp[target_col] == 0][col]
        g1 = temp[temp[target_col] == 1][col]

        if len(g0) < 2 or len(g1) < 2:
            continue

        # Mann–Whitney U test (비모수, 양측검정)
        u_stat, p_val = mannwhitneyu(g0, g1, alternative="two-sided")

        mean_0 = g0.mean()
        mean_1 = g1.mean()
        median_0 = g0.median()
        median_1 = g1.median()

        results.append({
            "feature": col, "target": target_col, "n_0": len(g0), "n_1": len(g1), "mean_0": mean_0, "mean_1": mean_1, "mean_diff": mean_1 - mean_0, "median_0": median_0, "median_1": median_1, "median_diff": median_1 - median_0, "u_stat": u_stat, "p_value": p_val
        })

    result_df = pd.DataFrame(results)

    def sig_label(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    result_df["significance"] = result_df["p_value"].apply(sig_label)
    result_df["is_significant_0.05"] = result_df["p_value"] < 0.05

    return result_df.sort_values("p_value", ascending=True).reset_index(drop=True)


# --------------------------------------------------
# 3. 표면 / 구조 각각 수행
# --------------------------------------------------
surface_mwu = run_mwu_all_features(train_eda, feature_cols, "표면")
structure_mwu = run_mwu_all_features(train_eda, feature_cols, "구조")

print("\n[표면 불량 기준 Mann–Whitney U test 결과]")
display(surface_mwu)

print("\n[구조 불량 기준 Mann–Whitney U test 결과]")
display(structure_mwu)


# --------------------------------------------------
# 4. 유의한 변수만 보기
# --------------------------------------------------
surface_sig = surface_mwu[surface_mwu["p_value"] < 0.05].copy()
structure_sig = structure_mwu[structure_mwu["p_value"] < 0.05].copy()

print("\n[표면 불량 기준 유의한 변수 (p < 0.05)]")
display(surface_sig)

print("\n[구조 불량 기준 유의한 변수 (p < 0.05)]")
display(structure_sig)


# --------------------------------------------------
# 표면/구조에서 공통적으로 유의한 변수
# --------------------------------------------------
surface_sig_features = set(surface_sig_sorted["feature"])
structure_sig_features = set(structure_sig_sorted["feature"])

common_features = sorted(surface_sig_features & structure_sig_features)

print("\n[표면/구조 공통적으로 유의한 변수]")
display(common_features)


# --------------------------------------------------
# 공통 유의 변수만 따로 보기
# --------------------------------------------------
surface_common = surface_sig_sorted[surface_sig_sorted["feature"].isin(common_features)].copy()
structure_common = structure_sig_sorted[structure_sig_sorted["feature"].isin(common_features)].copy()

print("\n[표면 불량 - 공통 유의 변수만]")
display(surface_common)

print("\n[구조 불량 - 공통 유의 변수만]")
display(structure_common)

# ANOVA

각 공정 변수에 대해

1. 값을 4구간으로 나눔 (quantile bin)
2. 각 구간의 불량 여부 평균 계산
3. 그룹 평균 차이에 대해 ANOVA 수행
4. p-value 기준으로 유의 변수 찾기

### 컬럼명
- `feature`: 분석한 변수 이름
- `target`: 기준이 된 불량 유형(표면/구조)
- `n_bins`: 해당 변수를 나눈 구간 개수
- `f_stat`: ANOVA 검정 통계량
- `p_value`: 구간별 불량률 차이의 유의확률
- `min_rate`: 가장 낮은 구간 불량률
- `max_rate`: 가장 높은 구간 불량률
- `rate_diff`: 최고 불량률과 최저 불량률의 차이
- `bin_1 ~ bin_4`: 변수 값을 나눈 각 구간 범위
- `rate_1 ~ rate_4`: 각 구간에서의 불량률
- `significance`: p-value를 별표로 표시한 유의수준
- `is_significant_0.05`: 0.05 기준 유의 여부

In [ ]:
from scipy.stats import f_oneway

print("\n" + "="*60)
print("전체 변수 자동 ANOVA")
print("="*60)




# --------------------------------------------------
# 2. 단일 타겟에 대한 ANOVA 함수
# --------------------------------------------------
def run_anova_all_features(df, feature_cols, target_col, q=4):
    results = []

    for col in feature_cols:
        temp = df[[col, target_col]].dropna().copy()

        # qcut으로 구간화
        try:
            temp["bin"] = pd.qcut(temp[col], q=q, duplicates="drop")
        except ValueError:
            continue

        # 구간 수가 2개 미만이면 skip
        if temp["bin"].nunique() < 2:
            continue

        # 각 구간별 target 값 추출
        groups = [g[target_col].values for _, g in temp.groupby("bin", observed=False)]

        # 그룹 크기 체크
        if len(groups) < 2:
            continue
        if any(len(g) < 2 for g in groups):
            continue

        # ANOVA
        f_stat, p_val = f_oneway(*groups)

        # 구간별 불량률
        bin_rate = temp.groupby("bin", observed=False)[target_col].mean()

        result_row = {
            "feature": col,
            "target": target_col,
            "n_bins": temp["bin"].nunique(),
            "f_stat": f_stat,
            "p_value": p_val,
            "min_rate": bin_rate.min(),
            "max_rate": bin_rate.max(),
            "rate_diff": bin_rate.max() - bin_rate.min()
        }

        # 각 구간 불량률도 컬럼으로 저장
        for i, (bin_name, rate) in enumerate(bin_rate.items(), start=1):
            result_row[f"bin_{i}"] = str(bin_name)
            result_row[f"rate_{i}"] = rate

        results.append(result_row)

    result_df = pd.DataFrame(results)

    def sig_label(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    result_df["significance"] = result_df["p_value"].apply(sig_label)
    result_df["is_significant_0.05"] = result_df["p_value"] < 0.05

    return result_df.sort_values(["p_value", "rate_diff"], ascending=[True, False]).reset_index(drop=True)


# --------------------------------------------------
# 3. 표면 / 구조 각각 수행
# --------------------------------------------------
surface_anova = run_anova_all_features(train_eda, feature_cols, "표면", q=4)
structure_anova = run_anova_all_features(train_eda, feature_cols, "구조", q=4)

print("\n[표면 불량 기준 ANOVA 결과]")
display(surface_anova)

print("\n[구조 불량 기준 ANOVA 결과]")
display(structure_anova)


# --------------------------------------------------
# 4. 유의한 변수만 보기
# --------------------------------------------------
surface_anova_sig = surface_anova[surface_anova["p_value"] < 0.05].copy()
structure_anova_sig = structure_anova[structure_anova["p_value"] < 0.05].copy()

print("\n[표면 불량 기준 유의한 변수 (p < 0.05)]")
display(surface_anova_sig)

print("\n[구조 불량 기준 유의한 변수 (p < 0.05)]")
display(structure_anova_sig)


# 공통 유의 변수
surface_anova_sig_sorted = surface_anova_sig.sort_values(
    ["rate_diff", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

structure_anova_sig_sorted = structure_anova_sig.sort_values(
    ["rate_diff", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

common_features = sorted(
    set(surface_anova_sig["feature"]) &
    set(structure_anova_sig["feature"])
)

print("\n[표면/구조 공통적으로 유의한 변수]")
display(common_features)

# **Type 2**

In [ ]:
# 결측치 확인
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': product_type_2.isnull().sum(),
    '결측비율(%)': (product_type_2.isnull().sum() / len(product_type_2) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")

In [ ]:
# 결측치 중앙값 대체
na_cols = product_type_2.columns[product_type_2.isnull().any()]
product_type_2[na_cols] = product_type_2[na_cols].fillna(product_type_2[na_cols].median())

print("="*60)
print("중앙값 대체 완료!!")
print("="*60)
print(f"결측치 데이터 수: {product_type_2.isnull().any().sum()}개")
print("="*60)

In [ ]:
product_type_2 = product_type_2.drop(columns={("defects","contamination_1"), ("defects","contamination_2"), ("defects","impurity_1"), ("defects","impurity_2"), ("defects","inclusions_2")})
product_type_2

In [ ]:
# Product_Type이 2인 행만 필터링하여 저장
product_type_2.to_csv('../data_processed/product_type_2.csv', index=False, encoding='utf-8-sig')

### Train / Test 분리

In [ ]:
defect_groups_2 = {
    "표면": ["dent_1", "dent_2", "stain_1", "scratch_1", "buring_mark_1"],
    "구조": ["short_shot_1", "short_shot_2", "blow_hole_1", "blow_hole_2", "bubble_1", "crack_1", "crack_2"]
}

X_train, X_test, y_train, y_test = get_train_test_data(product_type_2, defect_groups_2)

In [ ]:
print(X_train.info())
print("=" * 60)
print(y_train.info())

In [ ]:
plt.subplots(figsize=(30,30))
sns.heatmap(data=X_train.corr(),linewidth=0.1,annot=True,fmt='.2f',cmap='coolwarm')

# 단변량 분석
- 변수 하나의 분포/특성 파악


In [ ]:
print("=" * 60)
print("독립변수 분포 분석")
print("=" * 60)

import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

# 수치형 변수만 선택
num_cols = X_train.select_dtypes(include='number').columns

print(f"\n[수치형 변수 개수] {len(num_cols)}개")
print(num_cols.tolist())

# 각 변수의 기술통계 + 왜도 + 첨도 출력
summary_list = []

for col in num_cols:
    data = X_train[col].dropna()
    
    summary_list.append({
        '변수명': col,
        'count': data.count(),
        'mean': data.mean(),
        'std': data.std(),
        'min': data.min(),
        '25%': data.quantile(0.25),
        '50%': data.median(),
        '75%': data.quantile(0.75),
        'max': data.max(),
        '왜도': skew(data),
        '첨도': kurtosis(data)
    })

summary_df = pd.DataFrame(summary_list)
#print("\n[독립변수 통계 요약]")
#display(summary_df.round(3))

# 시각화
n_cols = 4   # 한 행에 변수 2개 (각 변수당 hist + box)
n_rows = math.ceil(len(num_cols) / (n_cols // 2))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

idx = 0

for col in num_cols:
    data = X_train[col].dropna()
    
    # 히스토그램
    sns.histplot(data, bins=30, ax=axes[idx], color='steelblue', kde=True)
    axes[idx].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5,
                      label=f"mean: {data.mean():.2f}")
    axes[idx].axvline(data.median(), color='green', linestyle='--', linewidth=1.5,
                      label=f"median: {data.median():.2f}")
    axes[idx].set_title(f"{col} hist\n왜도={skew(data):.2f}, 첨도={kurtosis(data):.2f}", fontsize=12)
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

    # 박스플롯
    sns.boxplot(x=data, ax=axes[idx + 1], color='orange')
    axes[idx + 1].set_title(f"{col} box", fontsize=12)
    axes[idx + 1].grid(True, alpha=0.3)

    idx += 2

# 남는 subplot 제거
for j in range(idx, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 왜도가 큰 변수

In [ ]:
skew_vals = X_train.skew().abs()

log_cols = skew_vals[skew_vals > 3].index

for c in log_cols:
    X_train[c] = np.log1p(X_train[c])
    X_test[c]  = np.log1p(X_test[c])

# 다변량 분석
- 변수 여러 개 간의 관계 분석


In [ ]:
print("\n" + "="*60)
print("EDA용 데이터 생성")
print("="*60)

train_eda = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
)

print("train_eda shape:", train_eda.shape)
display(train_eda.head())
print("\n컬럼 목록:")
print(train_eda.columns.tolist())

### 통계적 가설 검정 - t-test

In [ ]:
from scipy.stats import ttest_ind

print("\n" + "="*60)
print("전체 변수 자동 t-test")
print("="*60)

# --------------------------------------------------
# 1. EDA용 데이터 준비
# --------------------------------------------------
train_eda = pd.concat(
    [X_train.reset_index(drop=True), y_train.reset_index(drop=True)],
    axis=1
).copy()

# 수치형 독립변수 추출
all_num_cols = train_eda.select_dtypes(include="number").columns.tolist()
target_cols = ["표면", "구조"]
feature_cols = [c for c in all_num_cols if c not in target_cols]

print(f"분석 대상 변수 개수: {len(feature_cols)}")
print(feature_cols)


# --------------------------------------------------
# 2. 단일 타겟에 대한 t-test 함수
# --------------------------------------------------
def run_ttest_all_features(df, feature_cols, target_col):
    results = []

    for col in feature_cols:
        temp = df[[col, target_col]].dropna().copy()

        # 그룹 분리
        g0 = temp[temp[target_col] == 0][col]
        g1 = temp[temp[target_col] == 1][col]

        # 그룹 크기 체크
        if len(g0) < 2 or len(g1) < 2:
            continue

        # Welch's t-test (등분산 가정 안 함)
        t_stat, p_val = ttest_ind(g0, g1, equal_var=False)

        mean_0 = g0.mean()
        mean_1 = g1.mean()
        median_0 = g0.median()
        median_1 = g1.median()

        results.append({
            "feature": col,
            "target": target_col,
            "n_0": len(g0),
            "n_1": len(g1),
            "mean_0": mean_0,
            "mean_1": mean_1,
            "mean_diff": mean_1 - mean_0,
            "median_0": median_0,
            "median_1": median_1,
            "median_diff": median_1 - median_0,
            "t_stat": t_stat,
            "p_value": p_val
        })

    result_df = pd.DataFrame(results)

    # 유의성 라벨
    def sig_label(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    result_df["significance"] = result_df["p_value"].apply(sig_label)
    result_df["is_significant_0.05"] = result_df["p_value"] < 0.05

    return result_df.sort_values("p_value", ascending=True).reset_index(drop=True)


# --------------------------------------------------
# 3. 표면 / 구조 각각 수행
# --------------------------------------------------
surface_ttest = run_ttest_all_features(train_eda, feature_cols, "표면")
structure_ttest = run_ttest_all_features(train_eda, feature_cols, "구조")

print("\n[표면 불량 기준 t-test 결과]")
display(surface_ttest)

print("\n[구조 불량 기준 t-test 결과]")
display(structure_ttest)


# --------------------------------------------------
# 4. 유의한 변수만 보기
# --------------------------------------------------
surface_sig = surface_ttest[surface_ttest["p_value"] < 0.05].copy()
structure_sig = structure_ttest[structure_ttest["p_value"] < 0.05].copy()

print("\n[표면 불량 기준 유의한 변수 (p < 0.05)]")
display(surface_sig)

print("\n[구조 불량 기준 유의한 변수 (p < 0.05)]")
display(structure_sig)


# --------------------------------------------------
# 차이 큰 순으로 정렬
# mean_diff, median_diff 둘 다 고려
# --------------------------------------------------
surface_t_sig_sorted = surface_sig.sort_values(
    ["median_diff", "mean_diff", "p_value"],
    ascending=[False, False, True]
).reset_index(drop=True)

structure_t_sig_sorted = structure_sig.sort_values(
    ["median_diff", "mean_diff", "p_value"],
    ascending=[False, False, True]
).reset_index(drop=True)

# --------------------------------------------------
# 표면/구조에서 공통적으로 유의한 변수
# --------------------------------------------------
surface_sig_features = set(surface_t_sig_sorted["feature"])
structure_sig_features = set(structure_t_sig_sorted["feature"])

common_features = sorted(
    surface_sig_features & structure_sig_features
)

print("\n[표면/구조 공통적으로 유의한 변수]")
display(common_features)

##  통계적 가설 검정 - ANOVA

In [ ]:
from scipy.stats import f_oneway

print("\n" + "="*60)
print("전체 변수 자동 ANOVA")
print("="*60)




# --------------------------------------------------
# 2. 단일 타겟에 대한 ANOVA 함수
# --------------------------------------------------
def run_anova_all_features(df, feature_cols, target_col, q=4):
    results = []

    for col in feature_cols:
        temp = df[[col, target_col]].dropna().copy()

        # qcut으로 구간화
        try:
            temp["bin"] = pd.qcut(temp[col], q=q, duplicates="drop")
        except ValueError:
            continue

        # 구간 수가 2개 미만이면 skip
        if temp["bin"].nunique() < 2:
            continue

        # 각 구간별 target 값 추출
        groups = [g[target_col].values for _, g in temp.groupby("bin", observed=False)]

        # 그룹 크기 체크
        if len(groups) < 2:
            continue
        if any(len(g) < 2 for g in groups):
            continue

        # ANOVA
        f_stat, p_val = f_oneway(*groups)

        # 구간별 불량률
        bin_rate = temp.groupby("bin", observed=False)[target_col].mean()

        result_row = {
            "feature": col,
            "target": target_col,
            "n_bins": temp["bin"].nunique(),
            "f_stat": f_stat,
            "p_value": p_val,
            "min_rate": bin_rate.min(),
            "max_rate": bin_rate.max(),
            "rate_diff": bin_rate.max() - bin_rate.min()
        }

        # 각 구간 불량률도 컬럼으로 저장
        for i, (bin_name, rate) in enumerate(bin_rate.items(), start=1):
            result_row[f"bin_{i}"] = str(bin_name)
            result_row[f"rate_{i}"] = rate

        results.append(result_row)

    result_df = pd.DataFrame(results)

    def sig_label(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""

    result_df["significance"] = result_df["p_value"].apply(sig_label)
    result_df["is_significant_0.05"] = result_df["p_value"] < 0.05

    return result_df.sort_values(["p_value", "rate_diff"], ascending=[True, False]).reset_index(drop=True)


# --------------------------------------------------
# 3. 표면 / 구조 각각 수행
# --------------------------------------------------
surface_anova = run_anova_all_features(train_eda, feature_cols, "표면", q=4)
structure_anova = run_anova_all_features(train_eda, feature_cols, "구조", q=4)

print("\n[표면 불량 기준 ANOVA 결과]")
display(surface_anova)

print("\n[구조 불량 기준 ANOVA 결과]")
display(structure_anova)


# --------------------------------------------------
# 4. 유의한 변수만 보기
# --------------------------------------------------
surface_anova_sig = surface_anova[surface_anova["p_value"] < 0.05].copy()
structure_anova_sig = structure_anova[structure_anova["p_value"] < 0.05].copy()

print("\n[표면 불량 기준 유의한 변수 (p < 0.05)]")
display(surface_anova_sig)

print("\n[구조 불량 기준 유의한 변수 (p < 0.05)]")
display(structure_anova_sig)


# 공통 유의 변수
surface_anova_sig_sorted = surface_anova_sig.sort_values(
    ["rate_diff", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

structure_anova_sig_sorted = structure_anova_sig.sort_values(
    ["rate_diff", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

common_features = sorted(
    set(surface_anova_sig["feature"]) &
    set(structure_anova_sig["feature"])
)

print("\n[표면/구조 공통적으로 유의한 변수]")
display(common_features)